In [5]:
import pandas as pd
import numpy as np

from sklearn.metrics import classification_report, accuracy_score
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import ExtraTreesClassifier, GradientBoostingClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier


from sklearn.model_selection import GridSearchCV, KFold, train_test_split, RandomizedSearchCV

from imblearn.over_sampling import RandomOverSampler
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

from collections import Counter

In [4]:
data = pd.read_parquet("dataset_pred_genres_57.parquet", parse_dates=['release_date'])

data["release_date"] = pd.to_datetime(
    data["release_date"], format="%Y-%m-%d", errors="coerce"
)

data["month"] = data["release_date"].dt.month
data["dayofweek"] = data["release_date"].dt.dayofweek

data['month_sin'] = np.sin(2 * np.pi * data['month'] / 12)
data['month_cos'] = np.cos(2 * np.pi * data['month'] / 12)

data['dayofweek_sin'] = np.sin(2 * np.pi * data['dayofweek'] / 7)
data['dayofweek_cos'] = np.cos(2 * np.pi * data['dayofweek'] / 7)

data.drop(["release_date", "month", "dayofweek"], axis=1, inplace=True)

# Lista de variables a utilizar

In [5]:
features = [
    "new_popularity_2025",
    "acousticness",
    "danceability",
    "duration_ms",
    "energy",
    "explicit",
    "instrumentalness",
    "key",
    "liveness",
    "loudness",
    "mode",
    "speechiness",
    "tempo",
    "valence",
    "year",
    "month_cos",
    "month_sin",
    "dayofweek_cos",
    "dayofweek_sin",
    "featuring_cat"
]

target = "main_genre"

# Preparación de datos para modelo

In [25]:
X = data[features].copy()

# --- Preprocesamiento ---

# Aplicar LabelEncoder para convertir las clases en numéricas
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(data[target])

# Estandarizar las features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# División de datos

In [7]:
# --- División de datos ---
# Aquí usamos stratify=y para mantener la distribución de clases
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# --- Experimento 1: Balanced Training ---
min_count = 2000  # mínimo de muestras que quieres por clase
class_counts = Counter(y_train)
sampling_strategy = {
    cls: min_count for cls, count in class_counts.items() if count < min_count
}
ros = RandomOverSampler(sampling_strategy=sampling_strategy, random_state=42)

X_train_bal, y_train_bal = ros.fit_resample(X_train, y_train)

# Modelos

In [ ]:
# # --- Definición de los modelos ---
# models = {
#     'XGBoost': XGBClassifier(eval_metric='mlogloss', random_state=42),
#     'SVM': SVC(random_state=42),
#     'RandomForest': RandomForestClassifier(random_state=42)
# }

# # --- Entrenamiento y Evaluación ---
# for name, model in models.items():
#     print("\n=====================================")
#     print("Modelo:", name)
    
#     # Experimento 1: Entrenamiento balanceado
#     model.fit(X_train_bal, y_train_bal)
#     y_pred_bal = model.predict(X_test)
#     print("\n-- Balanced Training --")
#     print(classification_report(y_test, y_pred_bal))
    
#     # Experimento 2: Entrenamiento con partición relativa (sin balancear)
#     model.fit(X_train, y_train)
#     y_pred = model.predict(X_test)
#     print("\n-- Relative Partition Training --")
#     print(classification_report(y_test, y_pred))


In [17]:
# KFold para validación cruzada
kf = KFold(n_splits=3, shuffle=True, random_state=42)

# Diccionario de modelos y sus grids
model_grids = {
    "RandomForest": {
        "model": RandomForestClassifier(random_state=42),
        "params": {
            "n_estimators": [100, 200, 300],
            "max_depth": [None, 10, 20, 30],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 4],
            "bootstrap": [True, False],
            "class_weight": ["balanced", None]
        }
    },
    "XGBoost": {
        "model": XGBClassifier(eval_metric="mlogloss", random_state=42),
        "params": {
            "n_estimators": [100, 200, 300],
            "learning_rate": [0.01, 0.1, 0.3],
            "max_depth": [3, 6, 10],
            "subsample": [0.6, 0.8, 1.0],
            "colsample_bytree": [0.6, 0.8, 1.0],
            "gamma": [0, 1, 5],
            "reg_alpha": [0, 0.1, 1],
            "reg_lambda": [1, 5, 10]
        }
    },
    "LightGBM": {
        "model": LGBMClassifier(random_state=42),
        "params": {
            "n_estimators": [100, 200, 300],
            "learning_rate": [0.01, 0.1, 0.3],
            "max_depth": [-1, 10, 20],
            "num_leaves": [31, 64, 128],
            "min_child_samples": [20, 50, 100],
            "subsample": [0.6, 0.8, 1.0],
            "colsample_bytree": [0.6, 0.8, 1.0],
            "reg_alpha": [0, 0.1, 1],
            "reg_lambda": [1, 5, 10]
        }
    },
    "CatBoost": {
        "model": CatBoostClassifier(verbose=0, random_state=42),
        "params": {
            "iterations": [200, 500],
            "learning_rate": [0.01, 0.1, 0.3],
            "depth": [4, 6, 10],
            "l2_leaf_reg": [1, 3, 5, 7],
            "random_strength": [0.5, 1, 2]
        }
    },
    "SVM": {
        "model": SVC(random_state=42),
        "params": {
            "C": [0.1, 1, 10],
            "kernel": ["linear", "rbf", "poly"],
            "gamma": ["scale", "auto"],
            "degree": [2, 3, 4]
        }
    },
    "LogisticRegression": {
        "model": LogisticRegression(solver='saga', multi_class='multinomial', max_iter=1000, random_state=42),
        "params": {
            "C": [0.01, 0.1, 1, 10],
            "penalty": ["l2", "l1", "elasticnet"],
            "l1_ratio": [0.1, 0.5, 0.9]
        }
    },
    "KNN": {
        "model": KNeighborsClassifier(),
        "params": {
            "n_neighbors": [3, 5, 7, 11],
            "weights": ["uniform", "distance"],
            "metric": ["euclidean", "manhattan", "minkowski"]
        }
    },
    "MLP": {
        "model": MLPClassifier(max_iter=300, random_state=42),
        "params": {
            "hidden_layer_sizes": [(100,), (100,100), (150,100,50)],
            "activation": ["relu", "tanh"],
            "solver": ["adam", "sgd"],
            "learning_rate_init": [0.001, 0.01, 0.1],
            "alpha": [0.0001, 0.001, 0.01]
        }
    },
    "ExtraTrees": {
        "model": ExtraTreesClassifier(random_state=42),
        "params": {
            "n_estimators": [100, 200, 300],
            "max_depth": [None, 10, 20],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 4]
        }
    },
    "GradientBoosting": {
        "model": GradientBoostingClassifier(random_state=42),
        "params": {
            "n_estimators": [100, 200],
            "learning_rate": [0.01, 0.1, 0.3],
            "max_depth": [3, 5, 7],
            "subsample": [0.6, 0.8, 1.0]
        }
    }
}

resultados = []

for name, config in model_grids.items():
    print(f"\n=====================================\nModelo: {name}")

    grid = GridSearchCV(
        estimator=config["model"],
        param_grid=config["params"],
        scoring="accuracy",
        cv=kf,
        verbose=0,
        n_jobs=-1
    )

    # --------- Entrenamiento con datos balanceados ---------
    print("\n-- Balanced Training --")
    grid.fit(X_train_bal, y_train_bal)
    y_pred_bal = grid.predict(X_test)
    acc_bal = accuracy_score(y_test, y_pred_bal)

    resultados.append(
        {
            "modelo": name,
            "mejores_parametros": grid.best_params_,
            "accuracy": acc_bal,
        }
    )

    print("Mejores parámetros:", grid.best_params_)
    print("Accuracy:", acc_bal)
    print(classification_report(y_test, y_pred_bal))

df_resultados = pd.DataFrame(resultados)
df_resultados.to_csv("resultados_modelos.parquet", index=False)
print("\n✅ Resultados guardados en 'resultados_modelos.parquet'")


Modelo: XGBoost

-- Balanced Training --


/Users/artemmindlin/Library/CloudStorage/OneDrive-UPV/UPV/proyectos/3_c2/proyiii/ML/genres/.venv/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [08:44:27] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/artemmindlin/Library/CloudStorage/OneDrive-UPV/UPV/proyectos/3_c2/proyiii/ML/genres/.venv/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [08:44:27] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/artemmindlin/Library/CloudStorage/OneDrive-UPV/UPV/proyectos/3_c2/proyiii/ML/genres/.venv/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [08:44:27] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=

Mejores parámetros: {'learning_rate': 0.3, 'max_depth': 6, 'n_estimators': 200}
              precision    recall  f1-score   support

           0       0.10      0.02      0.03       225
           1       0.64      0.40      0.49       203
           2       0.61      0.36      0.45       303
           3       0.42      0.35      0.38       269
           4       0.37      0.36      0.37       289
           5       0.45      0.35      0.39       507
           6       0.48      0.24      0.32       673
           7       0.52      0.26      0.35       204
           8       0.41      0.39      0.40      1036
           9       0.31      0.04      0.07       109
          10       0.23      0.16      0.19       141
          11       0.18      0.03      0.06       123
          12       0.32      0.22      0.26       400
          13       0.17      0.06      0.09       340
          14       0.24      0.25      0.24       166
          15       0.17      0.12      0.14       215
 

/Users/artemmindlin/Library/CloudStorage/OneDrive-UPV/UPV/proyectos/3_c2/proyiii/ML/genres/.venv/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [08:57:24] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/artemmindlin/Library/CloudStorage/OneDrive-UPV/UPV/proyectos/3_c2/proyiii/ML/genres/.venv/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [08:57:24] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/artemmindlin/Library/CloudStorage/OneDrive-UPV/UPV/proyectos/3_c2/proyiii/ML/genres/.venv/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [08:57:24] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=

Mejores parámetros: {'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 200}
              precision    recall  f1-score   support

           0       0.33      0.01      0.02       225
           1       0.74      0.32      0.44       203
           2       0.64      0.31      0.42       303
           3       0.55      0.22      0.31       269
           4       0.36      0.38      0.37       289
           5       0.49      0.31      0.38       507
           6       0.62      0.15      0.24       673
           7       0.54      0.25      0.34       204
           8       0.45      0.38      0.41      1036
           9       0.56      0.05      0.08       109
          10       0.33      0.11      0.16       141
          11       0.00      0.00      0.00       123
          12       0.37      0.23      0.28       400
          13       0.21      0.02      0.03       340
          14       0.22      0.08      0.12       166
          15       0.29      0.08      0.13       215
 